# 07 · Safety consequences (Table 8)

**Runtime: A100** (L4 works: set `cfg["safety"]["batch_size"] = 16`). Needs both adapters (notebook 04), `HF_TOKEN` is not needed, `ANTHROPIC_API_KEY` in Colab secrets for the refusal classifier.

Benchmark (D6, pilot): **XSTest** (`Paul/XSTest`, CC-BY-4.0), translated with the study's MT model like the eval set. 200 unsafe prompts (a safe model refuses: **safety** = refusal rate) and 250 safe prompts that only look unsafe (a safe model answers: **over-refusal** = refusal rate). Report it as machine-translated XSTest.

| Step | Output on Drive (`MyDrive/mpac_hi_pt/safety/`) | A100 |
|---|---|---|
| 1 Translate XSTest | `translations/`, `items.jsonl` | ~5 min |
| 2 Generate (3 checkpoints × en/hi/pt, 256 new tokens) | `generations/` | ~20–30 min per checkpoint |
| 3 Classify (Claude Haiku: full_compliance / full_refusal / partial_refusal) + table | `classifier/`, `results/<model>/safety/table8.json` | ~10 min, < $1 |

Every step resumes: translations, generations and classifier replies are cached.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[gpu,llm]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
MODEL = "qwen2.5-7b"
CONFIGS = ["configs/hi_pt.yaml"]

import os, torch
from google.colab import drive, userdata
drive.mount("/content/drive")
from mpac import load_config
from mpac.config import get_model
from mpac.paths import resolve_root
cfg = load_config(*CONFIGS)
m = get_model(cfg, MODEL)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
if "A100" not in gpu and "H100" not in gpu:
    cfg["safety"]["batch_size"] = 16
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    print("ANTHROPIC_API_KEY not in Colab secrets: needed for step 3")
print("model:", m["id"], "| GPU:", gpu, "| artifacts ->", resolve_root(cfg))

## 1 · Translate XSTest

In [ ]:
import gc
from collections import Counter
from mpac.safety.evaluate import build_safety_items

items = build_safety_items(cfg)
gc.collect(); torch.cuda.empty_cache()   # frees MADLAD before generation
print(len(items), "items |", Counter(it.meta["label"] for it in items))
for it in items[:3]:
    print({k: v[:80] for k, v in it.prompts.items()})

## 2 · Generate, one checkpoint at a time

In [ ]:
from mpac.safety.evaluate import generate_safety

for ck in cfg["checkpoints"]:
    print("==", ck, generate_safety(cfg, m, ck, items))
    gc.collect(); torch.cuda.empty_cache()

## 3 · Classify and score

In [ ]:
import pandas as pd
from mpac.safety.evaluate import make_classifier, table8

clf = make_classifier(cfg)
t8 = table8(cfg, m["id"], clf, items=items)
print("saved ->", t8["path"], "| classifier:", t8["classifier"], "| missing:", t8["missing_generations"], "\n")
df = pd.DataFrame(t8["rows"])
cols = [c for c in ["checkpoint", "lang", "n_unsafe", "safety", "safety_full", "safety_vs_ref", "safety_p",
                    "n_safe", "over_refusal", "over_refusal_vs_ref", "over_refusal_p", "acs_forced", "acs_free"] if c in df]
print(df[cols].round(3).to_string(index=False), "\n")
print("ACS link (descriptive,", t8["n_points"], "points):", {k: round(v, 3) for k, v in t8["acs_link"].items()})

In [ ]:
# Spot-check the classifier: a few responses per label in Hindi (base)
from mpac.safety.evaluate import refusal_prompt, safety_cache
gens = safety_cache(cfg).load_all(m["id"], "base", ["hi"])["hi"]
by_id = {it.item_id: it for it in items}
sample = list(gens)[:6]
for iid, r in zip(sample, clf([refusal_prompt(by_id[i], "hi", gens[i].response) for i in sample])):
    print(iid, by_id[iid].meta["label"], "->", r.get("label"), "|", r.get("evidence"), "\n   ", gens[iid].response[:200].replace("\n", " "))